<a href="https://colab.research.google.com/github/ywang0202/NLP_Group_Project/blob/main/NLP_2026_Group_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NLP 2026 Group Project**

---

Authors: Hosein Mohebbi & Giovanni Cassani

---

This notebook is part of the Natural Language Processing (Spring 2026) course and is designed to replicate some of the experiments presented in [How Many Data Points is a Prompt Worth?](https://aclanthology.org/2021.naacl-main.208/)
The main goal of this project is to investigate two common approaches (head-based and prompt-based) to fine-tuning large language models for classification tasks.

## Introduction

Before everything, let's quickly go over some key terminology, including the main stages of LLM development: pre-training and fine-tuning, and two fine-tuning approaches: head-based and prompt-based, used throughout this project.

### __Pre-training__
Large language models (LLMs) are pre-trained using self-supervised learning objectives such as masked language modeling or next-token prediction over massive corpora.
Architecturally, a language modeling head (a linear classifier) followed by a softmax operation maps the final hidden representations (outputed from a stack of Transformer layers) to a probability distribution over the whole vocabulary. The training objective is to maximize the likelihood of the correct masked/next token given the context. Through this objective, the base model learns rich internal representations that capture broad, task-agnostic properties of language, including lexical semantics and syntactic structure. These pre-trained representations provide a good initialization that can be effectively adapted for various downstream applications.

### __Fine-tuning__
Fine-tuning means further training the base model for a few steps on supervised, task-specific data to help it adapt to a particular task. This adaptation is achieved by adjusting the model's parameters (all or partial weights) through gradient-based updates. In this project, we compare two commonly-used fine-tuning approaches:

__head-based__ tuning: a conventional approach where the base model is coupled with a newly initialized classification head and is fine-tuned to predict the categorized output class.

__prompt-based__ tuning: where we reformulate the input examples in the dataset as __cloze-style__ prompts, in which one token is intentionally masked and the model is asked to fill in the blank. For instance, consider the task of sentiment analysis, where the goal is to classify a review as either positive or negative. Given the input text:
> _Excellent pizza! Slices are fantastic, prices are reasonable._

Instead of training a classifier on top of the model to map the sentence representation to discrete output classes (e.g., 0 for positive and 1 for negative), prompt-based tuning reformulates the task as a cloze-style prediction problem. Specifically, the input could be rewritten as:
> Excellent pizza! Slices are fantastic, prices are reasonable. The restaurant is ___

The model is then asked to predict a word for the blank position. Ideally, it outputs "good" if the review is positive (class 0) and "bad" if the review is negative (class 1). In general, this approach requires two things. First, a __pattern__, which is a function that reformulates the original input into a cloze-style prompt. Second, a __verbalizer__, which maps each possible output class to a single token in the model's vocabulary.

Interestingly, the input and output format in __prompt-based__ fine-tuning approach aligns very well with the pre-training objective that is used to pre-train the base models (masked language modeling or next-token prediction). This makes this approach especially useful in low-resource settings, where only limited fine-tuning data is available as there is no need to train a new classifier from scratch: the approach leverages the already pre-trained language model head and repurposes it for classification. In this regards, the paper aims to compare the sample efficiency of the two approaches to answer _"how many data points is a prompt worth?"_
The following notebook is provided to help students fine-tune the models to replicate some of the experiments presented in the paper.

In [ ]:
# @title Install Requirements
!pip install transformers datasets evaluate

In [ ]:
# @title Imports required libraries
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import default_data_collator
from torch.optim import AdamW
from datasets import load_dataset
from transformers import RobertaTokenizer, RobertaConfig, RobertaForMaskedLM, RobertaForSequenceClassification
from evaluate import load as load_metric


In [ ]:
# @title Select GPU
if torch.cuda.is_available():
    device = torch.device(f"cuda:0")
    print('We will use the GPU:', torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print('No GPU available, using the CPU instead.')


## Models

Here, we load pre-trained weights of the RoBERTa model (pre-trained by the masked language modeling objective) using the HuggingFace's Transformers library.

__RobertaForSequenceClassification__ loads the pre-trained RoBERTa model and attaches a newly initialized classification head. This head applies a linear projection to the hidden state of the special \<s\> (beginning-of-sequence) token, producing logits whose dimensionality matches the number of labels in the fine-tuning dataset.

__RobertaForMaskedLM__ loads the pre-trained RoBERTa model together with its original language modeling head used during the pre-training stage. The language modeling head maps hidden states at masked positions to a probability distribution over the entire vocabulary to fill in the masked positions.

In [ ]:
# @title Load model
def load_model_tokenizer(model_path, method, device):
  tokenizer = RobertaTokenizer.from_pretrained(model_path)
  if method == "head":
      config = RobertaConfig.from_pretrained(model_path)
      config.num_labels = 3 # should be set to the number of labels in the target fine-tuning dataset
      model = RobertaForSequenceClassification.from_pretrained(model_path, config=config)
  else:
      model = RobertaForMaskedLM.from_pretrained(model_path)
  model.to(device)

  return model, tokenizer


## Preprocessing input examples

In this notebook, we provide the code for working with the __CB__ (__CommitmentBank__) task from the [SuperGLUE](https://super.gluebenchmark.com/) benchmark.

The CB task is a natural language inference problem in which, given two pieces of text, a __premise__ and a __hypothesis__, the model must determine their semantic relationship. Specifically, the model performs a 3-way classification, predicting whether the hypothesis is entailed by the premise (label 0), contradicted by the premise (label 1), or neutral with respect to the premise (label 2). To illustrate the task, consider the following example from the CB dataset (see more examples of the dataset [here](https://huggingface.co/datasets/aps/super_glue/viewer/cb?row=79)):

> __premise__: She didn't know if they had given themselves sufficient time to think things over before they married - that was the kind of question her sister Louise asked. Edward stayed in the Engineers for a bit, then came out and was not very successful in finding a job to suit him. That wasn't his fault and if anyone said that it was Nenna would still feel like poking a hole in them.

> __hypothesis__: it was Edward's fault

> __label__: contradiction

In this section, we prepare the input features for the model by converting the input text into token IDs.

For __head-based__ approach: it is standard to concatenate the _'premise'_ and _'hypothesis'_ with the special seperator token.
```
input: <s> She didn't know if they had given themselves sufficient time to think things over before they married - that was the kind of question her sister Louise asked. Edward stayed in the Engineers for a bit, then came out and was not very successful in finding a job to suit him. That wasn't his fault and if anyone said that it was Nenna would still feel like poking a hole in them. </s></s> it was Edward's fault </s>
label: 1
```

For __prompt-based__ approach, the input text can be reformulated to a cloze-style format using the following pattern:

___hypothesis?___ ___\<seperator\>___ ___<mask>,___ ___premise___

```
input: <s> it was Edward's fault? </s></s> <mask>, She didn't know if they had given themselves sufficient time to think things over before they married - that was the kind of question her sister Louise asked. Edward stayed in the Engineers for a bit, then came out and was not very successful in finding a job to suit him. That wasn't his fault and if anyone said that it was Nenna would still feel like poking a hole in them. </s>
label: no
```
The masked token prediction is mapped to a verbalizer. (here entailment: 'yes', contradiction: 'no', neutral: 'maybe').
For more details about the patterns and verbaliers, check out the Appendix A (Choice of prompts) in the paper.


In [ ]:
# @title Prepare input features (head-based)
def preprocessing_head(example, tokenizer):
    """
    prepares a single example as the input of the RoBERTa model for head-based fine-tuning.

    Inputs:
        example (dict):
            A dictionary containing:
                - 'premise' (str): The premise sentence.
                - 'hypothesis' (str): The hypothesis sentence.
                - 'label' (int): The ground-truth class label.
        tokenizer (PreTrainedTokenizer): RoBERTa's tokenizer used to encode the input text.

    Outputs:
        features (dict):
            A dictionary containing:
                - 'input_ids' (List[int]): Token IDs of the encoded input.
                - 'attention_mask' (List[int]): Attention mask distinguising actual vs. padded tokens, since different examples in a batch may have different sequence lengths.
                - 'labels' (int): The target class label.
    """

    features = tokenizer(example['premise'], example['hypothesis'], truncation='only_first', padding="max_length", max_length=MAXL_LENGTH)
    features['labels'] = example['label']
    return features


In [ ]:
# @title Prepare input features (prompt-based)
VERBALIZER = {0: 'yes', 1: 'no', 2: 'maybe'} # RoBERTa tokenizer always tokenizes these words into single tokens
def preprocessing_prompt(example, tokenizer):
    """
    Prepares a single example as the input of RoBERTa model for prompt-based fine-tuning.

    Inputs:
        example (dict):
            A dictionary containing:
                - 'premise' (str): The premise sentence.
                - 'hypothesis' (str): The hypothesis sentence.
                - 'label' (int): The ground-truth class label.
        tokenizer (PreTrainedTokenizer): RoBERTa's tokenizer used to encode the input text.

    Outputs:
        features (dict):
            A dictionary containing:
                - 'input_ids' (List[int]): Token IDs of the prompt-formatted input.
                - 'attention_mask' (List[int]): Attention mask distinguising actual vs. padded tokens, since different examples in a batch may have different sequence lengths.
                - 'masked_indices' (int): Index of the masked token position in input_ids.
                - 'labels' (int): The target class label.
    """
    max_len = MAXL_LENGTH
    pattern = f"{example['hypothesis']}?{tokenizer.sep_token}{tokenizer.sep_token}{tokenizer.mask_token},"
    partial = tokenizer.encode(pattern, add_special_tokens=False)
    premise = tokenizer.encode(" " + example['premise'], truncation=True, max_length=MAXL_LENGTH - len(partial) - 2, add_special_tokens=False)

    # input feture with padd and attention mask
    input_ids = [tokenizer.bos_token_id] + partial + premise + [tokenizer.eos_token_id]
    # truncate if needed
    input_ids = input_ids[:max_len]
    # pad if needed
    pad_len = max_len - len(input_ids)
    if pad_len > 0:
        input_ids = input_ids + [tokenizer.pad_token_id] * pad_len
    attention_mask = [1] * (max_len - pad_len) + [0] * pad_len

    # label (verbalizer)
    masked_index = np.where(np.array(input_ids) == tokenizer.mask_token_id)[0].item()
    label = example["label"]

    features = {}
    features['input_ids'] = input_ids
    features['attention_mask'] = attention_mask
    features['masked_indices'] = masked_index
    features['labels'] = label
    return features

## Evaluation
The following function evaluates a fine-tuned model on the test set by generating predictions, and computes the F1-macro evaluation metric based on the predicted and ground-truth labels.

In [ ]:
# @title Evaluation loop

def eval(model, tokenizer, test_dataloader, metric, device):
  predictions = []
  references = []
  model.eval()
  for batch in test_dataloader:
      batch = {k: v.to(device) for k, v in batch.items()}
      with torch.no_grad():
          outputs = model(batch["input_ids"], attention_mask=batch["attention_mask"])

      if METHOD == "head":
          preds = torch.argmax(outputs.logits, dim=-1)
      else:
          logits = outputs.logits
          masked_logits = logits[torch.arange(logits.size(0)), batch["masked_indices"]]
          target_ids = torch.tensor([tokenizer.convert_tokens_to_ids(v) for v in VERBALIZER.values()], device=device)
          logits_of_interest = masked_logits[:, target_ids]
          preds = torch.argmax(logits_of_interest, dim=-1)

      predictions.extend(preds)
      references.extend(batch["labels"])

  perf = metric.compute(predictions=predictions, references=references, average="macro")
  return perf

## Training
The following function implements the training loop for both head-based and prompt-based fine-tuning, computing the appropriate cross-entropy loss for each method, updating model parameters over multiple epochs, and reporting evaluation performance after each epoch.

In __prompt-based__ approach, the cross-entropy loss is computed over the restricted set of tokens defined by the verbalizer, rather than over the full vocabulary. For example, in the CB task, the objective is to maximize the probability assigned to the verbalized token corresponding to the ground-truth label among only the tokens _"yes"_, _"no"_, and _"maybe"_.

In [ ]:
# @title Training loop

def train(model, tokenizer, train_dataloader, test_dataloader, metric, epochs, device):
  total_training_steps = len(train_dataloader) * EPOCHS
  print(f"Starting training on {total_training_steps} number of training data points")

  global_step = 0
  for e in range(epochs):
      print(f"\n===== Epoch {e+1}/{epochs} =====")
      # Train
      model.train()
      for batch in train_dataloader:
          # forward pass
          batch = {k: v.to(device) for k, v in batch.items()}
          outputs = model(batch["input_ids"], attention_mask=batch["attention_mask"])

          if METHOD == "head":
            logits = outputs.logits
            labels = batch["labels"]
            loss = torch.nn.functional.cross_entropy(logits, labels)
          else:
            logits = outputs.logits
            masked_logits = logits[torch.arange(logits.size(0)), batch["masked_indices"]]
            target_ids = torch.tensor([tokenizer.convert_tokens_to_ids(v) for v in VERBALIZER.values()], device=device)
            logits_of_interest = masked_logits[:, target_ids]
            labels = batch["labels"]
            loss = torch.nn.functional.cross_entropy(logits_of_interest, labels)

          loss.backward()
          torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
          optimizer.step()
          optimizer.zero_grad()

          print(f"Step {global_step}/{total_training_steps}, CE Loss: {loss.item()}")
          global_step += 1

      # eval
      perf = eval(model=model, tokenizer=tokenizer, test_dataloader=test_dataloader, metric=metric, device=device)
      print(f"Test F1-macro at epoch #{e}: {perf}")

  return model, perf

## Running experiments

Here, you can run fine-tuning using the selected approach (head vs. prompt) and the specified number of training data points. Evaluation is always performed on a fixed number of held-out training examples.

In [ ]:
METHOD = "prompt" # @param ["head", "prompt"]
NUM_EXAMPLES = "16" # @param [8, 16, 24, 32, 48, 72, 96, 152, 200]
NUM_EXAMPLES = int(NUM_EXAMPLES)
EPOCHS = 20 # @param {type:"slider", min:1, max:200, step:1}
LEARNING_RATE = 1e-5
MODEL_PATH = "FacebookAI/roberta-base"
BATCH_SIZE = 8
MAXL_LENGTH = 92

# Load Model, Optimizer & Metric
model, tokenizer = load_model_tokenizer(model_path=MODEL_PATH, method=METHOD, device=device)
optimizer = AdamW(params=model.parameters(), lr=LEARNING_RATE)
metric = load_metric("f1")

# Load data
data = load_dataset("aps/super_glue", "cb", split="train").shuffle()
data = data.train_test_split(test_size=50, shuffle=True, seed=42)
train_data = data['train'].select(range(NUM_EXAMPLES))
test_data = data['test']

# Prepare input features
preprocess = preprocessing_prompt if METHOD == "prompt" else preprocessing_head
train_dataset = train_data.map(preprocess, fn_kwargs={"tokenizer": tokenizer}, remove_columns=train_data.column_names)
test_dataset = test_data.map(preprocess, fn_kwargs={"tokenizer": tokenizer}, remove_columns=test_data.column_names)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=default_data_collator, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=default_data_collator, pin_memory=True)

# Run training & eval
model, performance = train(model=model, tokenizer=tokenizer, train_dataloader=train_dataloader, test_dataloader=test_dataloader, metric=metric, epochs=EPOCHS, device=device)


# Assignments

## Task #1
Using the provided notebook, reproduce the second plot in Figure 1 of the paper for the CB task using the RoBERTa-base model (8 points total). The task consists of the following sub-tasks:

* __1.1.__ Run the experiment using the head-based finetuning with varying amounts of training data ([8, 16, 24, 32, 48, 72, 96, 152, 200]) and record the corresponding test performance. Set the appropriate hyper-parameters! (2 pts)

* __1.2.__ Run the experiment using the prompt-based fine-tuning with the same varying amounts of training data and record the corresponding test performance! (2 pts)

* __1.3__ Plot performance curves corresponding to the CB task shown in Figure 1 (you can leave visual descriptors concerning the advantage  and region of comparison out). Note that, for each data point, the paper reports the best performance that has been achieved at that amount of data or lower! (3 pts).

* __1.4__ Is prompt-based tuning feasible when the verbalizer labels are tokenized into multiple sub-tokens by the LM's tokenizer rather than a single token? If so, describe how it could work. (1 pts)

## Task #2
Repeat the same experimental procedure for the BoolQ task (corresponding to the second panel in Figure 1). To reduce computational overhead, restrict the training set size of BoolQ to a maximum of 2,000 data points. When loading the dataset and after shuffling it with seed=42, use the first 500 examples as the test set and the subsequent 2,000 examples as the training set (8 pts).
Points are awarded for the following sub-tasks:

* __2.1.__ Run the head-based classifier on the BoolQ task, at varying training data sizes ([8, 16, 24, 32, 48, 72, 96, 152, 200, 320, 504, 704, 800, 1000, 2000]) (2 pts)

* __2.3.__ Run the prompt-based classifier on the BoolQ task using "yes" or "no" as verbalizers for True and False, at varying data sizes ([8, 16, 24, 32, 48, 72, 96, 152, 200, 320, 504, 704, 800, 1000, 2000]) (2 pts)

* __2.3.__ Reproduce the plots (again you can leave visual descriptors concerning the advantage  and region of comparison out) (4 pts)

For this experiment, you will need to modify the number of output labels and adapt the preprocessing functions. For prompt-based tuning, use the first cloze-style pattern, specified in Appendix A in the paper;
Given a passage p and question q:

___p.___ ___Question:___ ___q?___ ___Answer:___ ___


__Note__: For both Tasks #1 and #2, you are allowed to run experiments multiple times and report the average results. You may also vary the number of training epochs. However, you are not allowed to change the random seed used for shuffling the original data, the learning rate, the target models, the pattern and verbalizer, or any other components of the experimental setup beyond these allowances.

## Task #3
This project introduces two approaches for fine-tuning language models: head-based fine-tuning and prompt-based fine-tuning, both of which update all model parameters using gradient updates. There are several alternative approaches for adapting LLMs to downstream tasks, including parameter-efficient fine-tuning (PEFT) methods (e.g., LoRA), as well as In-Context Learning (ICL) where task behavior is induced through few demonstrations in the input prompt at inference time.
Study about PEFT and ICL, to answer following sub-tasks (4 pts in total):

* __3.1.__ Highlight at least two advantages and two disadvantages of prompt-based tuning compared to head-based tuning. (2 pts)

* __3.2.__ Discuss the advantages of PEFT and ICL seperately compared to head-based and prompt-based fine-tuning approaches, especially when applied to very large-scale language models with billions of parameters. (2 pts)

__Note__: Please keep your answers clear and concise, with a maximum of 150 words for each sub-task. Any text beyond this limit will not be counted.